# RAG Basics: Retrieval-Augmented Generation

Reach for this when you need: 
- Reference for building LLM apps that search private documents.
- To implement Vector Database-style retrieval with FAISS.
- Understanding the bridge between retrieval (searching) and generation (answering).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel, pipeline
import numpy as np
import faiss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Vector Database with FAISS

FAISS (Facebook AI Similarity Search) is the industry standard for efficient dense vector storage and retrieval.

| Component | Description | Usage |
| :--- | :--- | :--- |
| `IndexFlatIP` | Inner Product index | High accuracy, slow search |
| `IndexHNSW` | Hierarchical Navigable Small World | Very fast, approximate search |
| `IndexIVF` | Inverted File Index | Partitioning data for speed |

In [ ]:
d = 768 # Embedding dimension
index = faiss.IndexFlatIP(d) # Simple dot product index

# Add dummy vectors to index
vectors = np.random.randn(10, d).astype('float32')
index.add(vectors)

# Search
query_vector = np.random.randn(1, d).astype('float32')
D, I = index.search(query_vector, k=3) # Top 3 nearest neighbors

## 2. Generating with Context

Passing retrieved fragments as part of the LLM prompt templates.

✅ **Use when**: Accurate, fact-grounded answering is needed (e.g. documentation bot).
❌ **Don't use when**: Broad creative writing is the goal; context can restrict model creativity.

In [ ]:
retrieved_context = "PyTorch was developed by Meta's AI Research lab (FAIR)."
user_query = "Who created PyTorch?"

prompt = f"Context: {retrieved_context}\nQuestion: {user_query}\nAnswer:"

# Inference
gen_pipeline = pipeline("text-generation", model="mistralai/Mistral-7B-Instruct-v0.2", device_map="auto")
output = gen_pipeline(prompt, max_new_tokens=50)

### Common Pitfalls
- **Normalization**: For `IndexFlatIP` to work correctly as Cosine Similarity, embeddings MUST be L2-normalized `v / norm(v)` before indexing and query.
- **Chunking**: Don't index whole documents. LLMs have limited context windows (e.g. 4096). Break docs into 500-token chunks with some overlap.
- **Metadata Loss**: FAISS only stores vectors. You MUST maintain a separate mapping from `index_id` -> `original_text`.

### Key Takeaways
- RAG scales an LLM's knowledge without expensive retraining.
- Retrieval is a similarity search in an embedding space.
- Embedding models (e.g. `all-MiniLM-L6-v2`) are separate from generation models.